# StainNormProgram Inputs

This tutorial demonstrates how to input images into StainNormProgram.

In [ ]:
from stainnormprogram.norm import Normalization

## Case 1
### One source WSI

Normalize one source WSI using one target WSI.

In [ ]:
N = Normalization()
N.set_target("TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs")
N.set_source("TCGA-LL-A6FP-01Z-00-DX1.6261398A-7288-4924-BBE2-FC1949256E40.svs")
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

## Case 2
### Multiple source WSIs

Normalize multiple source WSIs.

In [ ]:
N = Normalization()
N.set_target("TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs")

# Provide a list of paths
N.set_source([
    "TCGA-LL-A6FP-01Z-00-DX1.6261398A-7288-4924-BBE2-FC1949256E40.svs",
    "TCGA-LL-A6FR-01Z-00-DX1.107010FD-8EDD-406D-B4DE-DEC9E85576FF.svs",
])

# Provide a folder
N.set_source("./source_images/")

# Provide a list of folders
N.set_source([
    "./source_set1/",
    "./source_set2/",
    "./source_set3/",
])

N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

## Case 3
### Multiple target WSIs

The target style is calculated as an average across multiple WSIs.

In [ ]:
N = Normalization()
N.set_target([
    "TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs",
    "TCGA-64-1680-01Z-00-DX1.9CB14F20-E7B0-49A1-A327-404B3160FD65.svs",
    "TCGA-64-5815-01Z-00-DX1.1E3600B5-7E83-47BE-AC70-E0C2D623740B.svs",
])
N.set_source("TCGA-LL-A6FP-01Z-00-DX1.6261398A-7288-4924-BBE2-FC1949256E40.svs")
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

## Case 4
### Target style JSON

If you have an already calculated style (e.g., you calculated it in a prior session), you can load it in instead of re-computing it.

In [ ]:
N = Normalization()
N.set_target("target.json")
N.set_source("TCGA-LL-A6FP-01Z-00-DX1.6261398A-7288-4924-BBE2-FC1949256E40.svs")
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

## Case 5
### Pre- and post-adjustments

Before and after stain normalization, we can perform simple image adjustments. You might find yourself in this situation if the stain normalization method has certain failure modes.

For example, suppose your normalization method fails if the pixel values are too close to 0. Encountering such values may result in the normalization method producing artifacts. An "easy fix" might be to add a pre-adjustment that brightens the source images before normalization.

In [ ]:
from stainnormprogram.subset import Subset
from stainnormprogram.adjustments import Adjustment

In [ ]:
# First, we create a Subset with source images
subset = Subset("my_subset")
subset.add_images([
    "TCGA-LL-A6FP-01Z-00-DX1.6261398A-7288-4924-BBE2-FC1949256E40.svs",
    "TCGA-LL-A6FR-01Z-00-DX1.107010FD-8EDD-406D-B4DE-DEC9E85576FF.svs",
])

# Next, we create a pre-adjustment which brightens the image by adding the value 10 to all pixels
brighten = Adjustment(shift=10)

# Then, we add the pre-adjustment to the subset
subset.set_global_pre_adjustment(brighten)

# Finally, we run normalization using the usual functions but use the subset as the source
N = Normalization()
N.set_target("TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs")
N.set_source(subset)
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

## Case 6
### WSIs or tiles with a shared source reference

In this case, there are now three entities involved: (1) a target, (2) a source to be normalized, and (3) a source to use for the source style estimation.

In stain normalization, the staining style of your target distribution and the staining style of your source distribution are estimated so that we can perform the translation. Usually, we estimate the source style from the source image(s) and then stain normalize the same source image(s).

However, you may find yourself in the case where the source to estimate from and source to normalize aren't exactly the same.

In [ ]:
from stainnormprogram.subset import Subset

In [ ]:
"""
Suppose we want to use the average style across all source images as the source style instead of computing a different source style per image
"""

# First, create a subset with all source images
all_source_images = [
    "TCGA-LL-A50Y-01Z-00-DX1.7F08413F-2C59-4322-9296-84F8CD3DF619.svs",
    "TCGA-LL-A73Z-01Z-00-DX1.C010142E-29C0-411D-9E0E-4B7D8A4C09BF.svs",
    "TCGA-LL-A7SZ-01Z-00-DX1.4DAF6421-6A1D-41C8-BFD6-859FE10CB8CC.svs",
]
subset = Subset("source_images")
subset.add_images(all_source_images)

# Next, add the same images as reference
subset.add_reference(all_source_images)

# Finally, we run normalization using the usual functions but use the subset as the source
N = Normalization()
N.set_target("TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs")
N.set_source(subset)
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")

In [ ]:
"""
Suppose we calculated the source style of institution A in the past
Now, we have received more images from institution A and want to reuse the source style
"""

# First, create a subset with the new institution A images
subset = Subset("New_Institution_A_Images")
subset.add_images([
    "TCGA-LL-A50Y-01Z-00-DX1.7F08413F-2C59-4322-9296-84F8CD3DF619.svs",
    "TCGA-LL-A73Z-01Z-00-DX1.C010142E-29C0-411D-9E0E-4B7D8A4C09BF.svs",
    "TCGA-LL-A7SZ-01Z-00-DX1.4DAF6421-6A1D-41C8-BFD6-859FE10CB8CC.svs",
])

# Then, add the already computed source style as the reference
subset.add_reference("institution_A_style.json")

# Finally, we run normalization using the usual functions but use the subset as the source
N = Normalization()
N.set_target("TCGA-64-1677-01Z-00-DX1.7720D42B-1F57-4C23-9BC5-87C456E7737B.svs")
N.set_source(subset)
N.set_pipeline("MacenkoPipeline")
N.run_norm("./output/")